In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

import time

In [2]:
PROJECT_ROOT = (
    Path.cwd()
    .parent
)

SRC_DIR = (
    PROJECT_ROOT
    / "src"
)

if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )

In [3]:
from src.data_pipeline import (
    run_data_pipeline,
)

from src.supertrend_utils import (
    generate_supertrend,
)

from src.strategy_engine import (
    run_strategy_pipeline
)

from src.plotting_utils import (
    plot_ohlcv,
    plot_ohlcv_with_supertrend_bands,
    plot_ohlcv_with_supertrend,
    plot_strategy_trades,
    plot_cumulative_pnl,
    plot_strategy_performance,
    print_performance_summary,
)

In [4]:
# 1. Load and strictly validate all OHLCV data
# (Prints descriptive statistics and timestamp reports automatically)
ohlcv_data = run_data_pipeline(
    data_dir=Path("../data"),
    print_report=True
)

DATAFRAME INFORMATION
<class 'pandas.DataFrame'>
DatetimeIndex: 1165813 entries, 2023-03-20 14:31:00 to 2026-07-03 19:59:00
Data columns (total 6 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   open         1165813 non-null  float64
 1   high         1165813 non-null  float64
 2   low          1165813 non-null  float64
 3   close        1165813 non-null  float64
 4   tick_volume  1165813 non-null  int64  
 5   source_file  1165813 non-null  str    
dtypes: float64(4), int64(1), str(1)
memory usage: 62.3 MB

DESCRIPTIVE STATISTICS
               open          high           low         close
count  1.165813e+06  1.165813e+06  1.165813e+06  1.165813e+06
mean   2.957495e+03  2.958079e+03  2.956899e+03  2.957497e+03
std    9.822508e+02  9.826735e+02  9.818091e+02  9.822510e+02
min    1.811000e+03  1.811460e+03  1.810430e+03  1.811000e+03
25%    2.032830e+03  2.033050e+03  2.032600e+03  2.032830e+03
50%    2.647580e+03  2.647960e+03  2.

In [5]:
# 2. Generate Supertrend indicator and bands
# Tunable parameters: atr_period, multiplier, smoothing_type ("RMA", "SMA", "EMA")
supertrend_data = generate_supertrend(
    ohlcv_data=ohlcv_data,
    atr_period=14,
    multiplier=3.0,
    smoothing_type="RMA"
)

# Quick check on output
supertrend_data[["close", "supertrend", "trend", "atr"]].tail()

,close,supertrend,trend,atr
time,,,,
2026-07-03 19:55:00,4175.72,4171.844521,1.0,1.141826
2026-07-03 19:56:00,4176.46,4172.564913,1.0,1.131696
2026-07-03 19:57:00,4174.02,4172.564913,1.0,1.241575
2026-07-03 19:58:00,4175.60,4172.564913,1.0,1.281462
2026-07-03 19:59:00,4174.94,4172.564913,1.0,1.262072


In [21]:
# 3. Execute the full strategy pipeline
signal_data, trades_data, metrics, plot_data = run_strategy_pipeline(
    supertrend_data=supertrend_data,
    
    # --- Shared Parameters ---
    tick_size=0.01,
    initial_capital=1000000.0,
    trading_days_per_year=252,
    
    # --- Signal Generation Parameters ---
    penetration_model="atr_fraction",             # "ticks" or "atr_fraction"
    penetration_ticks=1,
    penetration_atr_fraction=2.2,          # Used if penetration_model="atr_fraction"
    sensitivity_scalar=1.0,
    squashing_type="tanh",                 # "tanh" or "relu"
    volume_multiplier=1.5,                 # Minimum volume spike threshold
    volume_ma_period=20,
    
    # --- Backtest & Risk Management Parameters ---
    reward_risk=5.0,                       # Fixed target R:R ratio
    stop_loss_ratio=1.0,                   # Multiplier for stop distance
    position_sizing="weighted",            # "fixed" or "weighted"
    max_position_per_trade_fraction=0.10,  # Capital allocated per trade
    max_risk_per_trade_fraction=0.01,
    min_holdings_fraction=0.80,            # Liquidation threshold (80% of initial)
    transaction_costs_model="percentage",  # "percentage" or "flat"
    transaction_costs_fraction=0.0005,     # 5 bps per leg
    transaction_costs_flat=2.50,
    same_bar_priority="stop",              # "stop" or "target"
    slippage_ticks=0,
    
    # --- Performance Parameters ---
    risk_free_rate=0.02,
)

In [22]:
print_performance_summary(metrics)

          STRATEGY PERFORMANCE TEAR SHEET

[ TRADE STATISTICS ]
--------------------------------------------------
Total Trades:           49
Win Rate:               24.49%
Profit Factor:          1.23
Average R-Multiple:     -0.42R

[ ABSOLUTE RETURNS ]
--------------------------------------------------
Total Net PnL:          $2,647.79
Total Return:           0.26%
Annualized Return:      0.08%

[ RISK & RISK-ADJUSTED METRICS ]
--------------------------------------------------
Maximum Drawdown ($):   $4,951.80
Maximum Drawdown (%):   0.49%
Annualized Sharpe:      -6.66
Annualized Sortino:     -16.45
Calmar Ratio:           0.16

